In [1]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("default")

def log(msg):
    print(msg, flush=True)
    sys.stdout.flush()

log("Imports OK.")


Imports OK.


In [2]:
# ==============================================================================
# 2. CONFIGURATION
# ==============================================================================

SEED = 42
TARGET_COL = "ausencia"

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DATA_PATH    = Path("..") / "data" / "processed" / "dados_uteis_refreshed_v3_memorysafe.pkl"
MODEL_SEARCH_DIR = Path("..") / "data" / "processed"
OUTPUT_DIR       = Path("..") / "data" / "processed" / "appendix"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model identity ─────────────────────────────────────────────────────────────
# This matches the RUN_TAG used in 07_random_forest_smote.ipynb
MODEL_RUN_TAG = "rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained"

# Set MODEL_PATH manually if auto-discovery fails.
# Prefer best_outer_artifacts_*.pkl (contains preprocessor + calibrator + threshold).
# Example:
#   MODEL_PATH = Path("../data/processed/best_outer_artifacts_rf_smote_refreshed_v3_...pkl")
MODEL_PATH = None   # None → auto-discover

# ── Sample ────────────────────────────────────────────────────────────────────
SAMPLE_SIZE     = 5_000     # number of observations in the demonstration sample
SAMPLE_STRATEGY = "proportional"   # "proportional" | "balanced"
# proportional: minority/majority shares match the full dataset (realistic)
# balanced:     50/50 class split (easier to read confusion matrix)

# ── Demo output tag — change to run a new version without overwriting old files ──
DEMO_TAG = "demo_rf_smote_v1"

# ── Fallback threshold (used only if artifact does not contain one) ───────────
DEFAULT_THRESHOLD = 0.140839

# ── Derived paths ─────────────────────────────────────────────────────────────
SAMPLE_PATH      = OUTPUT_DIR / f"sample_{DEMO_TAG}_{SAMPLE_SIZE}.pkl"
PREDICTIONS_PATH = OUTPUT_DIR / f"predictions_{DEMO_TAG}_{SAMPLE_SIZE}.csv"
METRICS_PATH     = OUTPUT_DIR / f"metrics_{DEMO_TAG}_{SAMPLE_SIZE}.json"
CONFUSION_PATH   = OUTPUT_DIR / f"confusion_matrix_{DEMO_TAG}_{SAMPLE_SIZE}.csv"
PREVIEW_PATH     = OUTPUT_DIR / f"preview_{DEMO_TAG}_{SAMPLE_SIZE}.csv"
OVERWRITE        = False   # set True to re-run and overwrite existing outputs

# ── Columns dropped before model input (must match training notebook) ─────────
DROP_COLS = [
    "cns_usuario", "ibge_usuario", "cep_usuario", "cep_cnes_exe",
    "ibge_cnes_exe", "lat_cnes", "lon_cnes", "lat_usuario", "lon_usuario",
    "cep_aprox_usuario", "cep_aprox_cnes", "dta_atend", "cod_uf_usuario",
    "cod_uf_cnes", "cidade_usuario", "cidade_cnes",
]

log("Configuration:")
log(f"  RAW_DATA_PATH : {RAW_DATA_PATH}")
log(f"  MODEL_RUN_TAG : {MODEL_RUN_TAG}")
log(f"  DEMO_TAG      : {DEMO_TAG}")
log(f"  SAMPLE_SIZE   : {SAMPLE_SIZE:,}")
log(f"  SAMPLE_STRATEGY: {SAMPLE_STRATEGY}")
log(f"  OUTPUT_DIR    : {OUTPUT_DIR}")


Configuration:
  RAW_DATA_PATH : ../data/processed/dados_uteis_refreshed_v3_memorysafe.pkl
  MODEL_RUN_TAG : rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained
  DEMO_TAG      : demo_rf_smote_v1
  SAMPLE_SIZE   : 5,000
  SAMPLE_STRATEGY: proportional
  OUTPUT_DIR    : ../data/processed/appendix


In [3]:
# ==============================================================================
# 3. SAFETY — no accidental overwriting
# ==============================================================================

def assert_can_write(path: Path, overwrite: bool = OVERWRITE) -> None:
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"File already exists and OVERWRITE=False:\n  {path}\n"
            "Change DEMO_TAG or set OVERWRITE=True."
        )

for p in [SAMPLE_PATH, PREDICTIONS_PATH, METRICS_PATH, CONFUSION_PATH, PREVIEW_PATH]:
    assert_can_write(p)

log("Output paths are safe — no existing files will be overwritten.")


Output paths are safe — no existing files will be overwritten.


In [4]:
# ==============================================================================
# 4. METRIC AND PIPELINE HELPERS
# ==============================================================================

def ensure_int_labels(y):
    arr = y.to_numpy() if isinstance(y, pd.Series) else np.asarray(y)
    return arr.ravel().astype(np.int64)

def binary_confusion(y_true, y_pred):
    cm = confusion_matrix(ensure_int_labels(y_true),
                          ensure_int_labels(y_pred), labels=[0, 1])
    if cm.shape != (2, 2):
        padded = np.zeros((2, 2), dtype=int)
        padded[:cm.shape[0], :cm.shape[1]] = cm
        cm = padded
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def full_metrics(y_true, y_pred, y_proba, threshold):
    y_true = ensure_int_labels(y_true)
    y_pred = ensure_int_labels(y_pred)
    tn, fp, fn, tp = binary_confusion(y_true, y_pred)
    total    = len(y_true)
    pred_pos = tp + fp
    pred_neg = tn + fn
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))
    spec = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
    npv  = float(tn / (tn + fn)) if (tn + fn) > 0 else float("nan")
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    acc  = float(accuracy_score(y_true, y_pred))
    try:
        auc_roc = float(roc_auc_score(y_true, y_proba))
    except ValueError:
        auc_roc = float("nan")
    youden_j = (rec + spec - 1.0) if not (
        np.isnan(rec) or np.isnan(spec)) else float("nan")
    bal_acc  = (rec + spec) / 2.0 if not (
        np.isnan(rec) or np.isnan(spec)) else float("nan")
    return {
        "Threshold":              float(threshold),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Real_Positive":          int((y_true == 1).sum()),
        "Real_Negative":          int((y_true == 0).sum()),
        "Predicted_Positive":     int(pred_pos),
        "Predicted_Negative":     int(pred_neg),
        "Predicted_Positive_Rate":  pred_pos / total if total else float("nan"),
        "Predicted_Negative_Rate":  pred_neg / total if total else float("nan"),
        "Accuracy":       acc,
        "Precision":      prec,
        "Recall":         rec,
        "Specificity":    spec,
        "NPV":            npv,
        "F1":             f1,
        "Youden_J":       youden_j,
        "Balanced_Accuracy": bal_acc,
        "ROC_AUC":        auc_roc,
    }

def print_metrics(m: dict):
    lines = [
        f"\n{'='*60}",
        f"  DEMO METRICS  (threshold = {m['Threshold']:.6f})",
        f"{'='*60}",
        f"  Confusion matrix:  [[TN={m['TN']:,}, FP={m['FP']:,}],",
        f"                       [FN={m['FN']:,}, TP={m['TP']:,}]]",
        f"  Real positive (no-show): {m['Real_Positive']:,} "
            f"({100*m['Real_Positive']/(m['Real_Positive']+m['Real_Negative']):.2f}%)",
        f"  Predicted positive:      {m['Predicted_Positive']:,} "
            f"({100*m['Predicted_Positive_Rate']:.2f}%)",
        f"  Predicted negative:      {m['Predicted_Negative']:,} "
            f"({100*m['Predicted_Negative_Rate']:.2f}%)",
        f"{'─'*60}",
        f"  Accuracy     : {m['Accuracy']:.6f}",
        f"  Precision    : {m['Precision']:.6f}",
        f"  Recall       : {m['Recall']:.6f}",
        f"  Specificity  : {m['Specificity']:.6f}",
        f"  NPV          : {m['NPV']:.6f}",
        f"  F1           : {m['F1']:.6f}",
        f"  Youden J     : {m['Youden_J']:.6f}",
        f"  Balanced Acc : {m['Balanced_Accuracy']:.6f}",
        f"  ROC AUC      : {m['ROC_AUC']:.6f}",
        f"{'='*60}",
    ]
    log("\n".join(lines))

# ── Calibration helpers (match training notebook exactly) ─────────────────────

def clip_proba(p, eps=1e-6):
    return np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)

def logit(p):
    p = clip_proba(p)
    return np.log(p / (1.0 - p))

def apply_platt_calibrator(calibrator, raw_proba: np.ndarray) -> np.ndarray:
    """Apply the Platt sigmoid calibrator saved in the artifact."""
    if calibrator is None:
        return np.asarray(raw_proba, dtype=np.float64)
    raw_proba = np.nan_to_num(np.asarray(raw_proba, float),
                               nan=0.5, posinf=1-1e-9, neginf=1e-9)
    z = logit(raw_proba).reshape(-1, 1)
    return calibrator.predict_proba(z)[:, 1]

log("Helpers defined.")


Helpers defined.


In [5]:
# ==============================================================================
# 5. LOAD RAW DATA AND CREATE STRATIFIED DEMONSTRATION SAMPLE
# ==============================================================================

log(f"Loading raw dataset from:\n  {RAW_DATA_PATH}")
t0 = time.time()
df_full = pd.read_pickle(RAW_DATA_PATH)
log(f"Loaded in {time.time()-t0:.1f}s | shape: {df_full.shape}")

if TARGET_COL not in df_full.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found. "
                   f"Available: {list(df_full.columns[:10])}")

y_full  = ensure_int_labels(df_full[TARGET_COL])
c0_full = int((y_full == 0).sum())
c1_full = int((y_full == 1).sum())
prevalence = c1_full / len(y_full)
log(f"Full dataset | total={len(y_full):,} | "
    f"class-0 (show)={c0_full:,} | class-1 (no-show)={c1_full:,} | "
    f"prevalence={prevalence:.4%}")

# ── Build the sample ──────────────────────────────────────────────────────────
if SAMPLE_PATH.exists() and not OVERWRITE:
    log(f"Sample file already exists — loading:\n  {SAMPLE_PATH}")
    sample_df = pd.read_pickle(SAMPLE_PATH)
    log(f"Loaded sample | shape: {sample_df.shape}")
else:
    log(f"Creating new sample | size={SAMPLE_SIZE:,} | strategy={SAMPLE_STRATEGY}")
    all_idx = np.arange(len(df_full))

    if SAMPLE_STRATEGY == "balanced":
        # 50/50 split — easier to read confusion matrix
        idx0 = np.where(y_full == 0)[0]
        idx1 = np.where(y_full == 1)[0]
        n_each = SAMPLE_SIZE // 2
        rng = np.random.default_rng(SEED)
        sampled0 = rng.choice(idx0, size=min(n_each, len(idx0)), replace=False)
        sampled1 = rng.choice(idx1, size=min(n_each, len(idx1)), replace=False)
        sampled_idx = np.concatenate([sampled0, sampled1])
        rng.shuffle(sampled_idx)
    else:
        # proportional — class shares match full dataset (realistic)
        _, sampled_idx = train_test_split(
            all_idx, test_size=min(SAMPLE_SIZE, len(all_idx)),
            stratify=y_full, random_state=SEED)

    sample_df = df_full.iloc[sampled_idx].copy().reset_index(drop=True)
    sample_df.to_pickle(SAMPLE_PATH)
    log(f"Sample saved: {SAMPLE_PATH}")

y_sample = ensure_int_labels(sample_df[TARGET_COL])
c0_s = int((y_sample == 0).sum())
c1_s = int((y_sample == 1).sum())
log(f"Sample | total={len(y_sample):,} | "
    f"class-0={c0_s:,} ({100*c0_s/len(y_sample):.1f}%) | "
    f"class-1={c1_s:,} ({100*c1_s/len(y_sample):.1f}%)")

X_raw = sample_df.drop(columns=[TARGET_COL]).copy()
y_true = y_sample
log(f"X_raw shape: {X_raw.shape}")


Loading raw dataset from:
  ../data/processed/dados_uteis_refreshed_v3_memorysafe.pkl
Loaded in 9.0s | shape: (7113798, 42)
Full dataset | total=7,113,798 | class-0 (show)=6,069,245 | class-1 (no-show)=1,044,553 | prevalence=14.6835%
Creating new sample | size=5,000 | strategy=proportional
Sample saved: ../data/processed/appendix/sample_demo_rf_smote_v1_5000.pkl
Sample | total=5,000 | class-0=4,266 (85.3%) | class-1=734 (14.7%)
X_raw shape: (5000, 41)


In [6]:
# ==============================================================================
# 6. LOCATE AND LOAD THE TRAINED MODEL ARTIFACT
# ==============================================================================

def find_best_artifact(model_run_tag: str, search_dir: Path):
    """
    Search for the best artifact file matching model_run_tag.
    Priority order:
      1. best_outer_artifacts_*.pkl  (full dict: model + preprocessor + calibrator + threshold)
      2. artifact_RF_*.pkl / artifact_outerfold_*.pkl
      3. best_outer_model_*.pkl      (model estimator only — no preprocessor)
    """
    priority_patterns = [
        f"best_outer_artifacts_{model_run_tag}.pkl",
        f"*artifacts*{model_run_tag}*.pkl",
        f"artifact_RF_{model_run_tag}*.pkl",
        f"artifact_outerfold_*_{model_run_tag}.pkl",
        f"*artifact*{model_run_tag}*.pkl",
        f"best_outer_model_{model_run_tag}.pkl",
        f"*model*{model_run_tag}*.pkl",
        f"*{model_run_tag}*.pkl",
    ]
    for pattern in priority_patterns:
        matches = sorted(search_dir.glob(pattern),
                         key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            return matches[0], pattern
    return None, None

if MODEL_PATH is not None:
    artifact_path = Path(MODEL_PATH)
    pattern_used  = "manual"
else:
    artifact_path, pattern_used = find_best_artifact(MODEL_RUN_TAG, MODEL_SEARCH_DIR)

if artifact_path is None or not artifact_path.exists():
    raise FileNotFoundError(
        f"Could not find a model artifact for RUN_TAG='{MODEL_RUN_TAG}'\n"
        f"Searched in: {MODEL_SEARCH_DIR}\n"
        "Set MODEL_PATH manually in the configuration cell."
    )

log(f"Found artifact:\n  {artifact_path}\n  (matched pattern: {pattern_used})")
raw_artifact = joblib.load(artifact_path)
log(f"Artifact type: {type(raw_artifact).__name__}")
if isinstance(raw_artifact, dict):
    log(f"Artifact keys: {list(raw_artifact.keys())}")

# ── Also try to load the config JSON for documentation ────────────────────────
config_path = MODEL_SEARCH_DIR / f"config_summary_{MODEL_RUN_TAG}.json"
config_meta = {}
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config_meta = json.load(f)
    log(f"Config metadata loaded: {config_path.name}")
else:
    log("Config JSON not found — will use defaults.")


Found artifact:
  ../data/processed/best_outer_artifacts_rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.pkl
  (matched pattern: best_outer_artifacts_rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.pkl)
Artifact type: dict
Artifact keys: ['Outer_Fold', 'Model', 'Preprocessor', 'Calibrator', 'Calibration_Method', 'Selected_Threshold', 'Threshold_Objective', 'Threshold_Row', 'X_val_sel', 'y_val', 'Feature_Names_Selected', 'Selected_Feature_Indices', 'Best_Params']
Config metadata loaded: config_summary_rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.json


In [7]:
# ==============================================================================
# 7. NORMALIZE ARTIFACT — extract model, preprocessor, calibrator, threshold
# ==============================================================================
# The training notebook saves artifacts as a dict with uppercase keys.
# This cell handles both that format and a bare estimator.

def _get(d, *keys, default=None):
    """Return the first matching key from a dict; return default if none found."""
    if not isinstance(d, dict):
        return default
    for k in keys:
        if k in d:
            return d[k]
    return default

if isinstance(raw_artifact, dict):
    model         = _get(raw_artifact, "Model", "model", "Estimator", "estimator")
    preprocessor  = _get(raw_artifact, "Preprocessor", "preprocessor",
                          "ColumnTransformer", "column_transformer")
    calibrator    = _get(raw_artifact, "Calibrator", "calibrator",
                          "Probability_Calibrator")
    feat_names    = _get(raw_artifact, "Selected_Feature_Names", "selected_feature_names",
                          "Feature_Names_Selected", "feature_names_selected",
                          "Selected_Features", "selected_features")
    feat_indices  = _get(raw_artifact, "Selected_Feature_Indices", "selected_feature_indices",
                          "Feature_Indices", "feature_indices")
    threshold     = _get(raw_artifact, "Selected_Threshold", "selected_threshold",
                          "Threshold", "threshold")
    threshold_row = _get(raw_artifact, "Threshold_Row", "threshold_row")
    cal_method    = _get(raw_artifact, "Calibration_Method", "calibration_method",
                          default="unknown")
else:
    # Bare estimator — preprocessor and calibrator must come from config or defaults
    model        = raw_artifact
    preprocessor = None
    calibrator   = None
    feat_names   = None
    feat_indices = None
    threshold    = None
    threshold_row= None
    cal_method   = "none"

if model is None:
    raise ValueError(
        "Could not extract the model from the artifact.\n"
        f"Artifact type: {type(raw_artifact)}\n"
        f"Keys: {list(raw_artifact.keys()) if isinstance(raw_artifact, dict) else 'N/A'}"
    )

# ── Coerce containers ─────────────────────────────────────────────────────────
if feat_names is not None and not isinstance(feat_names, list):
    feat_names = list(feat_names)
if feat_indices is not None:
    feat_indices = np.asarray(feat_indices, dtype=int)

# ── Resolve threshold ─────────────────────────────────────────────────────────
if threshold is None:
    # Try config metadata as fallback
    for k in ["selected_threshold", "threshold", "Selected_Threshold", "final_threshold"]:
        if k in config_meta:
            try:
                threshold = float(config_meta[k]); break
            except Exception:
                pass
if threshold is None:
    threshold = DEFAULT_THRESHOLD
    log(f"WARNING: threshold not found in artifact or config. "
        f"Using default = {threshold:.6f}")
threshold = float(threshold)

log("\nNORMALIZED ARTIFACT CONTENTS")
log("=" * 60)
log(f"  Model type          : {type(model).__name__}")
log(f"  Preprocessor type   : {type(preprocessor).__name__ if preprocessor else 'None'}")
log(f"  Calibrator type     : {type(calibrator).__name__ if calibrator else 'None'}")
log(f"  Calibration method  : {cal_method}")
log(f"  Selected features   : {len(feat_names) if feat_names else 'None'}")
log(f"  Selected indices    : {len(feat_indices) if feat_indices is not None else 'None'}")
log(f"  Threshold           : {threshold:.6f}")
if feat_names:
    log(f"  First 5 features    : {feat_names[:5]}")
if threshold_row:
    tr = threshold_row
    log(f"  Threshold tune stats: Youden-J={tr.get('Youden_J', 'N/A'):.4f}  "
        f"Recall={tr.get('Recall', 'N/A'):.4f}  "
        f"Specificity={tr.get('Specificity', 'N/A'):.4f}")



NORMALIZED ARTIFACT CONTENTS
  Model type          : RandomForestClassifier
  Preprocessor type   : ColumnTransformer
  Calibrator type     : LogisticRegression
  Calibration method  : sigmoid_platt_on_outer_calibration
  Selected features   : 20
  Selected indices    : 20
  Threshold           : 0.134224
  First 5 features    : ['tempo_espera', 'percentual_faltas_anteriores', 'contagem_faltas_anteriores', 'total_consultas_anteriores', 'idade_usuario']
  Threshold tune stats: Youden-J=0.3049  Recall=0.7201  Specificity=0.5848


In [8]:
# ==============================================================================
# 8. PREPROCESS INPUT MATRIX
# ==============================================================================
# Mirrors exactly what happens inside the outer fold of the training notebook:
#   1. Drop administrative/geo columns
#   2. Apply the fitted ColumnTransformer (MinMaxScaler + OHE)
#   3. Select the top-20 features by saved index

# ── Step 1: Drop columns ──────────────────────────────────────────────────────
cols_to_drop = [c for c in DROP_COLS if c in X_raw.columns]
X_input = X_raw.drop(columns=cols_to_drop).copy()
log(f"Step 1 | Dropped {len(cols_to_drop)} columns. "
    f"X_input shape: {X_input.shape}")

X_transformed_named = None   # will hold DataFrame with transformed feature names

# ── Step 2: Apply preprocessor ───────────────────────────────────────────────
if preprocessor is not None:
    log("Step 2 | Applying ColumnTransformer (MinMaxScaler + OHE)...")
    X_transformed = preprocessor.transform(X_input)
    # Try to recover feature names for alignment / debug output
    try:
        raw_names = preprocessor.get_feature_names_out().tolist()
        clean_names = [n.replace("num__","").replace("cat__","") for n in raw_names]
        X_transformed_named = pd.DataFrame(
            X_transformed, columns=clean_names, index=X_input.index)
        log(f"  Transformed shape: {X_transformed_named.shape} | "
            f"first 5 features: {clean_names[:5]}")
    except Exception:
        X_transformed_named = None
        log(f"  Transformed shape: {X_transformed.shape} "
            "(feature names not recoverable)")
else:
    log("Step 2 | No preprocessor in artifact. "
        "Assuming X_input is already in the correct numeric format.")
    X_transformed = X_input.to_numpy() if isinstance(X_input, pd.DataFrame) else X_input
    log(f"  Shape: {X_transformed.shape}")

X_for_prediction = X_transformed

# ── Step 3: Select top-20 features ───────────────────────────────────────────
if feat_names is not None and X_transformed_named is not None:
    # Prefer name-based selection (more robust to minor schema differences)
    missing = [n for n in feat_names if n not in X_transformed_named.columns]
    if not missing:
        X_for_prediction = X_transformed_named[feat_names].to_numpy()
        log(f"Step 3 | Selected {len(feat_names)} features by name. "
            f"Shape: {X_for_prediction.shape}")
    else:
        log(f"Step 3 | WARNING: {len(missing)} selected feature names missing "
            f"from preprocessor output. Falling back to index-based selection.")
        if feat_indices is not None:
            X_for_prediction = np.asarray(X_transformed)[:, feat_indices]
            log(f"  Index-based selection. Shape: {X_for_prediction.shape}")
        else:
            log("  No feature indices either — using all transformed features.")
elif feat_indices is not None:
    X_for_prediction = np.asarray(X_transformed)[:, feat_indices]
    log(f"Step 3 | Selected {len(feat_indices)} features by index. "
        f"Shape: {X_for_prediction.shape}")
else:
    log("Step 3 | No feature selection info — using all transformed features.")

X_for_prediction = np.asarray(X_for_prediction, dtype=np.float32)

# ── Align to model.feature_names_in_ if available ────────────────────────────
if (hasattr(model, "feature_names_in_") and
        X_transformed_named is not None and
        isinstance(X_for_prediction, np.ndarray) and
        X_for_prediction.shape[1] == len(model.feature_names_in_)):
    pass   # already aligned by the selection above
elif hasattr(model, "feature_names_in_") and X_transformed_named is not None:
    mf = list(model.feature_names_in_)
    if all(f in X_transformed_named.columns for f in mf):
        X_for_prediction = X_transformed_named[mf].to_numpy().astype(np.float32)
        log(f"  Re-aligned to model.feature_names_in_ ({len(mf)} features).")

log(f"\nFinal prediction input shape: {X_for_prediction.shape}")


Step 1 | Dropped 16 columns. X_input shape: (5000, 25)
Step 2 | Applying ColumnTransformer (MinMaxScaler + OHE)...
  Transformed shape: (5000, 179) | first 5 features: ['idade_usuario', 'tempo_espera', 'contagem_faltas_anteriores', 'total_consultas_anteriores', 'percentual_faltas_anteriores']
Step 3 | Selected 20 features by name. Shape: (5000, 20)

Final prediction input shape: (5000, 20)


In [9]:
# ==============================================================================
# 9. GENERATE PROBABILITIES AND APPLY THRESHOLD
# ==============================================================================

log("Generating raw probabilities...")
t0 = time.time()

# ── Raw probability from the RF ───────────────────────────────────────────────
if hasattr(model, "predict_proba"):
    raw_proba = model.predict_proba(X_for_prediction)[:, 1]
elif hasattr(model, "decision_function"):
    # SVM / SGD fallback: convert decision scores with sigmoid
    scores    = model.decision_function(X_for_prediction)
    raw_proba = 1.0 / (1.0 + np.exp(-scores))
else:
    log("WARNING: model has no predict_proba or decision_function. "
        "Using predict() as hard-label probabilities (0.0 / 1.0).")
    raw_proba = model.predict(X_for_prediction).astype(float)

raw_proba = np.nan_to_num(np.asarray(raw_proba, float),
                           nan=0.5, posinf=1-1e-9, neginf=1e-9)
log(f"  Raw proba: min={raw_proba.min():.6f}  "
    f"mean={raw_proba.mean():.6f}  max={raw_proba.max():.6f}  "
    f"time={time.time()-t0:.1f}s")

# ── Platt sigmoid calibration ─────────────────────────────────────────────────
if calibrator is not None:
    log("Applying Platt sigmoid calibrator (logit → LogisticRegression)...")
    cal_proba = apply_platt_calibrator(calibrator, raw_proba)
    log(f"  Calibrated proba: min={cal_proba.min():.6f}  "
        f"mean={cal_proba.mean():.6f}  max={cal_proba.max():.6f}")
else:
    cal_proba = raw_proba
    log("No calibrator in artifact — using raw probabilities directly.")

# ── Apply threshold ───────────────────────────────────────────────────────────
y_pred = (cal_proba >= threshold).astype(np.int32)

log(f"\nThreshold applied: {threshold:.6f}")
log(f"Predicted positive (no-show): {int(y_pred.sum()):,} "
    f"({100*y_pred.mean():.2f}%)")
log(f"Predicted negative (show):    {int((y_pred==0).sum()):,} "
    f"({100*(y_pred==0).mean():.2f}%)")


Generating raw probabilities...
  Raw proba: min=0.001557  mean=0.240079  max=0.809277  time=0.1s
Applying Platt sigmoid calibrator (logit → LogisticRegression)...
  Calibrated proba: min=0.001030  mean=0.146980  max=0.661917

Threshold applied: 0.134224
Predicted positive (no-show): 2,316 (46.32%)
Predicted negative (show):    2,684 (53.68%)


In [10]:
# ==============================================================================
# 10. PROBABILITY DISTRIBUTION ANALYSIS
# ==============================================================================
# Examine whether the model assigns meaningfully different scores to the two
# classes, and how the threshold sits relative to the score distributions.

proba_class0 = cal_proba[y_true == 0]
proba_class1 = cal_proba[y_true == 1]

log("PROBABILITY DISTRIBUTION BY TRUE CLASS")
log("=" * 60)
log(f"  Class 0 (show)    | n={len(proba_class0):,} | "
    f"mean={proba_class0.mean():.4f} | "
    f"median={np.median(proba_class0):.4f} | "
    f"p90={np.percentile(proba_class0, 90):.4f}")
log(f"  Class 1 (no-show) | n={len(proba_class1):,} | "
    f"mean={proba_class1.mean():.4f} | "
    f"median={np.median(proba_class1):.4f} | "
    f"p10={np.percentile(proba_class1, 10):.4f}")
log(f"  Threshold         : {threshold:.6f}")
log(f"  Above threshold   : {(cal_proba >= threshold).sum():,} "
    f"({100*(cal_proba >= threshold).mean():.2f}%)")

# ── Separation score: P(score >= thr | no-show) - P(score >= thr | show) ──────
tpr_at_thr = float((proba_class1 >= threshold).mean())
fpr_at_thr = float((proba_class0 >= threshold).mean())
log(f"\n  Recall at threshold    : {tpr_at_thr:.4f} "
    f"(fraction of true no-shows captured)")
log(f"  FP rate at threshold   : {fpr_at_thr:.4f} "
    f"(fraction of true shows flagged)")
log(f"  Separation (TPR-FPR)   : {tpr_at_thr - fpr_at_thr:.4f}")

# ── Percentile of threshold in each class distribution ─────────────────────
pct_in_c0 = float(np.mean(proba_class0 < threshold)) * 100
pct_in_c1 = float(np.mean(proba_class1 < threshold)) * 100
log(f"\n  Threshold sits above {pct_in_c0:.1f}% of class-0 scores "
    f"(correctly below threshold → TN)")
log(f"  Threshold sits above {pct_in_c1:.1f}% of class-1 scores "
    f"(below threshold → FN / missed no-shows)")


PROBABILITY DISTRIBUTION BY TRUE CLASS
  Class 0 (show)    | n=4,266 | mean=0.1363 | median=0.1166 | p90=0.2616
  Class 1 (no-show) | n=734 | mean=0.2090 | median=0.1932 | p10=0.0835
  Threshold         : 0.134224
  Above threshold   : 2,316 (46.32%)

  Recall at threshold    : 0.7384 (fraction of true no-shows captured)
  FP rate at threshold   : 0.4158 (fraction of true shows flagged)
  Separation (TPR-FPR)   : 0.3226

  Threshold sits above 58.4% of class-0 scores (correctly below threshold → TN)
  Threshold sits above 26.2% of class-1 scores (below threshold → FN / missed no-shows)


In [11]:
# ==============================================================================
# 11. EVALUATE — FULL METRICS AND CONFUSION MATRIX
# ==============================================================================

metrics = full_metrics(y_true, y_pred, cal_proba, threshold)
metrics.update({
    "sample_size":        int(len(y_true)),
    "sample_strategy":    SAMPLE_STRATEGY,
    "demo_tag":           DEMO_TAG,
    "model_run_tag":      MODEL_RUN_TAG,
    "calibration_method": cal_method,
    "artifact_path":      str(artifact_path),
    "n_selected_features": int(len(feat_names)) if feat_names else None,
    "selected_features":  feat_names,
})

print_metrics(metrics)

# ── Pretty confusion matrix DataFrame ────────────────────────────────────────
cm_df = pd.DataFrame(
    [
        [metrics["TN"], metrics["FP"]],
        [metrics["FN"], metrics["TP"]],
    ],
    index=pd.Index(["Real: Show (0)", "Real: No-show (1)"], name=""),
    columns=["Pred: Show (0)", "Pred: No-show (1)"],
)
log("\nCONFUSION MATRIX")
log("=" * 50)
print(cm_df.to_string())



  DEMO METRICS  (threshold = 0.134224)
  Confusion matrix:  [[TN=2,492, FP=1,774],
                       [FN=192, TP=542]]
  Real positive (no-show): 734 (14.68%)
  Predicted positive:      2,316 (46.32%)
  Predicted negative:      2,684 (53.68%)
────────────────────────────────────────────────────────────
  Accuracy     : 0.606800
  Precision    : 0.234024
  Recall       : 0.738420
  Specificity  : 0.584154
  NPV          : 0.928465
  F1           : 0.355410
  Youden J     : 0.322573
  Balanced Acc : 0.661287
  ROC AUC      : 0.712207

CONFUSION MATRIX
                   Pred: Show (0)  Pred: No-show (1)
                                                    
Real: Show (0)               2492               1774
Real: No-show (1)             192                542


In [12]:
# ==============================================================================
# 12. FEATURE IMPORTANCE (from the trained RandomForest)
# ==============================================================================
# The RF stores Gini importance for all its input features (the top-20 selected).
# This shows which of the 20 features the RF found most discriminative.

if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    names = feat_names if feat_names else [f"feature_{i}" for i in range(len(importances))]

    if len(names) == len(importances):
        df_imp = pd.DataFrame({
            "Feature":    names,
            "Importance": importances,
        }).sort_values("Importance", ascending=False).reset_index(drop=True)
        df_imp["Rank"]              = np.arange(1, len(df_imp) + 1)
        df_imp["Cumulative_Importance"] = df_imp["Importance"].cumsum()

        log("TOP-20 FEATURE IMPORTANCE (Gini, from RandomForestClassifier)")
        log("=" * 60)
        for _, row in df_imp.iterrows():
            bar = "█" * int(row["Importance"] / df_imp["Importance"].max() * 30)
            log(f"  {int(row['Rank']):2d}. {row['Feature']:<45s} "
                f"{row['Importance']:.6f}  {bar}")

        log(f"\n  Top-5 features account for "
            f"{df_imp.head(5)['Importance'].sum()*100:.1f}% of total importance.")
        log(f"  Top-10 features account for "
            f"{df_imp.head(10)['Importance'].sum()*100:.1f}% of total importance.")
    else:
        log(f"Feature importance shape mismatch: "
            f"{len(importances)} importances vs {len(names)} names.")
        df_imp = pd.DataFrame({"Feature": names[:len(importances)],
                                "Importance": importances})
else:
    log("Model does not expose feature_importances_ — skipping this section.")
    df_imp = pd.DataFrame()


TOP-20 FEATURE IMPORTANCE (Gini, from RandomForestClassifier)
   1. total_consultas_anteriores                    0.199804  ██████████████████████████████
   2. tempo_espera                                  0.192451  ████████████████████████████
   3. contagem_faltas_anteriores                    0.124809  ██████████████████
   4. mes_consulta                                  0.116924  █████████████████
   5. idade_usuario                                 0.074572  ███████████
   6. percentual_faltas_anteriores                  0.062278  █████████
   7. distance_km                                   0.051185  ███████
   8. mesma_unidade_anterior                        0.043516  ██████
   9. cep_prefix_len_cnes                           0.025696  ███
  10. cbo_espec_225265                              0.015793  ██
  11. tip_marcacao_2                                0.015203  ██
  12. cbo_espec_223710                              0.012634  █
  13. prioridade_3                              

In [13]:
# ==============================================================================
# 13. SAVE ALL OUTPUTS
# ==============================================================================

# ── Prediction CSV ────────────────────────────────────────────────────────────
pred_df = sample_df.copy()
pred_df["prob_no_show_raw"]        = raw_proba
pred_df["prob_no_show_calibrated"] = cal_proba
pred_df["predicted_no_show"]       = y_pred
pred_df["threshold_used"]          = threshold
pred_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8")
log(f"Predictions saved: {PREDICTIONS_PATH}")

# ── Confusion matrix CSV ──────────────────────────────────────────────────────
cm_df.to_csv(CONFUSION_PATH, encoding="utf-8")
log(f"Confusion matrix saved: {CONFUSION_PATH}")

# ── Metrics JSON ──────────────────────────────────────────────────────────────
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2, default=str)
log(f"Metrics saved: {METRICS_PATH}")

# ── Preview CSV (first 30 rows, key columns only) ─────────────────────────────
preview_cols = [
    TARGET_COL,
    "prob_no_show_raw",
    "prob_no_show_calibrated",
    "predicted_no_show",
    "threshold_used",
]
pred_df[preview_cols].head(30).to_csv(PREVIEW_PATH, index=False, encoding="utf-8")
log(f"Preview saved: {PREVIEW_PATH}")

# ── Feature importance CSV (if available) ─────────────────────────────────────
if not df_imp.empty:
    imp_path = OUTPUT_DIR / f"feature_importance_{DEMO_TAG}.csv"
    df_imp.to_csv(imp_path, index=False, encoding="utf-8")
    log(f"Feature importance saved: {imp_path}")

log("\nAll outputs saved successfully.")


Predictions saved: ../data/processed/appendix/predictions_demo_rf_smote_v1_5000.csv
Confusion matrix saved: ../data/processed/appendix/confusion_matrix_demo_rf_smote_v1_5000.csv
Metrics saved: ../data/processed/appendix/metrics_demo_rf_smote_v1_5000.json
Preview saved: ../data/processed/appendix/preview_demo_rf_smote_v1_5000.csv
Feature importance saved: ../data/processed/appendix/feature_importance_demo_rf_smote_v1.csv

All outputs saved successfully.


In [14]:
# ==============================================================================
# 14. FINAL SUMMARY
# ==============================================================================

log("\n" + "=" * 70)
log("  RF + SMOTE DEMONSTRATION — FINAL SUMMARY")
log("=" * 70)
log(f"  Model     : RandomForest + SMOTE (nested 10-fold CV)")
log(f"  Run tag   : {MODEL_RUN_TAG}")
log(f"  Artifact  : {artifact_path.name}")
log(f"  Sample    : {len(y_true):,} obs  "
    f"| {SAMPLE_STRATEGY} sampling | prevalence={100*y_true.mean():.2f}%")
log(f"  Threshold : {threshold:.6f}  (Youden-J)")
log(f"  Calibration: {cal_method}")
log("")
log(f"  ┌─────────────────────────────────────────┐")
log(f"  │  Metric           Value                 │")
log(f"  ├─────────────────────────────────────────┤")
log(f"  │  Accuracy       : {metrics['Accuracy']:.6f}               │")
log(f"  │  Precision      : {metrics['Precision']:.6f}               │")
log(f"  │  Recall         : {metrics['Recall']:.6f}               │")
log(f"  │  Specificity    : {metrics['Specificity']:.6f}               │")
log(f"  │  NPV            : {metrics['NPV']:.6f}               │")
log(f"  │  F1             : {metrics['F1']:.6f}               │")
log(f"  │  Youden J       : {metrics['Youden_J']:.6f}               │")
log(f"  │  Balanced Acc   : {metrics['Balanced_Accuracy']:.6f}               │")
log(f"  │  ROC AUC        : {metrics['ROC_AUC']:.6f}               │")
log(f"  ├─────────────────────────────────────────┤")
log(f"  │  TP={metrics['TP']:,}  TN={metrics['TN']:,}  "
    f"FP={metrics['FP']:,}  FN={metrics['FN']:,}  │")
log(f"  └─────────────────────────────────────────┘")
log("")
log("  Saved files:")
log(f"    {PREDICTIONS_PATH}")
log(f"    {METRICS_PATH}")
log(f"    {CONFUSION_PATH}")
log(f"    {PREVIEW_PATH}")
log("=" * 70)
log("Demonstration complete.")



  RF + SMOTE DEMONSTRATION — FINAL SUMMARY
  Model     : RandomForest + SMOTE (nested 10-fold CV)
  Run tag   : rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained
  Artifact  : best_outer_artifacts_rf_smote_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.pkl
  Sample    : 5,000 obs  | proportional sampling | prevalence=14.68%
  Threshold : 0.134224  (Youden-J)
  Calibration: sigmoid_platt_on_outer_calibration

  ┌─────────────────────────────────────────┐
  │  Metric           Value                 │
  ├─────────────────────────────────────────┤
  │  Accuracy       : 0.606800               │
  │  Precision      : 0.234024               │
  │  Recall         : 0.738420               │
  │  Specificity    : 0.584154               │
  │  NPV            : 0.928465               │
  │  F1             : 0.355410               │
  │  Youden J       : 0.322573               │
  │  Balanced Acc   : 0.661287               │
  │  ROC AUC        : 0.712207         

## How to re-run with a different sample

To generate a new demonstration sample without overwriting the existing one,
change the `DEMO_TAG` variable in **cell 2** — for example:

```python
DEMO_TAG = "demo_rf_smote_v2"
```

Then re-run all cells. Every output file name includes the tag, so previous
results are preserved.

## How to switch between proportional and balanced sampling

```python
SAMPLE_STRATEGY = "balanced"    # 50/50 class split
SAMPLE_STRATEGY = "proportional"  # mirrors full-dataset prevalence (default)
```

## About the inference pipeline

The full pipeline applied in this notebook is identical to what happens inside
each outer validation fold during training:

| Step | Code | Notes |
|---|---|---|
| Drop admin cols | `X_raw.drop(DROP_COLS)` | coordinates, IDs, dates |
| MinMax + OHE | `preprocessor.transform(X)` | fitted on training data only |
| Feature selection | `X[:, Selected_Feature_Indices]` | top-20 by permutation importance |
| RF probabilities | `model.predict_proba(X)[:, 1]` | class-1 = no-show |
| Platt calibration | `sigmoid_calibrator(logit(p))` | LR fitted on monitor set |
| Threshold | `p >= Selected_Threshold` | Youden-J optimal ≈ 0.14 |
